[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb)


# Occupational Classification: Survey Data and Model Evaluation

This notebook examines survey data and metadata, tests field-specific preparation decisions, and evaluates occupational classification models. Each transformation and modeling decision is tied to observable evidence and assessed against its alternatives.


In [2]:
# Standard library and notebook-wide utilities
import io
import hashlib
import urllib.request
import re
import json
import warnings
from pathlib import Path
import importlib.util
import subprocess
import sys
from urllib.request import urlopen


# Keep the previous warning policy scoped to UserWarning, not all warnings.
warnings.filterwarnings('ignore', category=UserWarning)

# Numerical arrays and tabular survey data
import numpy as np
import pandas as pd

# Notebook display configuration: improve readability without changing data.
pd.set_option('display.html.use_mathjax', False)
pd.set_option('display.max_columns', None)

def custom_style(df):
    """Format DataFrames for notebook display; source values remain unchanged."""
    return (df.style.set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#1e293b'), ('color', '#ffffff'),
            ('font-family', 'sans-serif'), ('font-size', '13px'),
            ('padding', '10px 15px'), ('text-align', 'left')
        ]},
        {'selector': 'td', 'props': [
            ('font-family', 'sans-serif'), ('font-size', '12px'),
            ('padding', '8px 15px'), ('border-bottom', '1px solid #e2e8f0')
        ]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f1f5f9')]}
    ]).set_properties(**{'background-color': '#f8fafc'}))

# Apply the HTML style to all DataFrame previews in this notebook session.
pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

# Figures and rich notebook output
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Machine learning: cross-validation and reusable pipelines
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Candidate classifiers and majority-class baseline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

# Performance measurement
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Reproducibility: downstream model evaluation uses this fixed random seed.
SEED = 42


### Load the LaBSE model

[LaBSE](https://huggingface.co/sentence-transformers/LaBSE?utm_source=chatgpt.com)

LaBSE (Language-agnostic BERT Sentence Embedding) is a multilingual model that converts text into 768-dimensional, L2-normalized vectors in a shared space. It was optimized to place translations close together, so we use it to compare workplace names written in Hebrew and English without translating them first. Because the vectors have unit length, their dot product equals cosine similarity. A high score identifies a candidate match between two workplace names. Establishing that both names refer to the same employer requires corroborating information.

The code below loads the pretrained model on the CPU. In the workplace-name analysis, we use cosine similarity to rank possible Hebrew matches for names written in English. A pair goes forward for review when its cosine similarity exceeds the adjustable threshold α. Confirming that both names refer to the same workplace requires separate evidence. 


In [31]:
# import sentence-transformers if not already installed
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2", ])
from sentence_transformers import SentenceTransformer

# Download/cache LaBSE once. No API token, SSH, or paid GPU.
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
model.eval()
print("LaBSE loaded.")

LaBSE loaded.


## Source data

Load the survey responses and their metadata from the two workbook sheets without altering the source values.


In [32]:
# Retrieve the original workbook, verify its contents, and load both source tables.
URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
SHA = "e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac"
raw = urllib.request.urlopen(URL, timeout=90).read()

# Load only rows and columns without missing values
data = (pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all"))

metadata = (pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any"))

# Display the complete metadata dictionary.
display(metadata)
print()

# Display the first three survey records.
display(data.head(20))
print()

,משתנה,הסבר,פורמט
0,ID,מספר רץ,INTEGER
1,YeshuvAvoda,שם יישוב עבודה,TEXT
2,ShemAvoda,שם מקום העבודה שלך,TEXT
3,SugAvoda,הפעילות העיקרית של מקום העבודה,TEXT
4,ShemMachlaka,אגף או מחלקה שהנסקר עובד,TEXT
5,SugMachlaka,הפעילות העיקרית של האגף או המחלקה שהנסקר עובד,TEXT
6,EzoAvoda,עבודה שהנסקר עושה במקום עבודה זה,TEXT
7,TeurPeula,הפעולות העיקריות שהנסקר עושה במקום עבודה,TEXT
8,TeurTafkid,מהו תואר המשרה של הנסקר במקום עבודה זה (במידה ויש),TEXT
9,MenahelEtMi,האם הנסקר מנהל או ממונה על עובדים אחרים?,INTEGER


,ID,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi
0,1.000000,ירושלים,"למ""ס","איסוף, עיבוד ופרסום נתונים",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל אגף,"כנ""ל",סגן מנהל אגף,3.000000,1.000000,1.000000,6.000000,22.000000,55.000000,2020-06-15 00:00:00,8411,2120
1,2.000000,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan,1.000000,1.000000,6.000000,22.000000,55.000000,2022-07-01 00:00:00,8610,3311
2,3.000000,באר שבע,קריית הממשלה,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,nan,nan,1.000000,1.000000,5.000000,15.000000,29.000000,2021-06-06 00:00:00,8411,3512
3,4.000000,תל אביב,משרד החינוך,בית ספר יסודי,שמירה,nan,שומר בבית הספר,nan,nan,nan,1.000000,2.000000,3.000000,12.000000,48.000000,2024-07-01 00:00:00,8010,5414
4,5.000000,תל אביב,אמיר כהן,מטפל בקשיש במשפחה,nan,nan,מטפל בקשיש,"נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ",nan,nan,1.000000,1.000000,4.000000,14.000000,42.000000,2010-01-11 00:00:00,XXXX,5322
5,6.000000,חיפה,משטרה,שוטר,nan,nan,שוטר,nan,nan,nan,1.000000,1.000000,4.000000,14.000000,35.000000,2022-05-31 00:00:00,8423,5412
6,7.000000,קרית גת,אינטל,ייצור צ'יפים למחשבים,חדר נקי,nan,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,nan,nan,1.000000,1.000000,5.000000,15.000000,65.000000,1999-07-07 00:00:00,2610,3122
7,8.000000,חיפה,אינטל,פיתוח תוכנה לצ'יפים,הנדסת תוכנה,nan,"מהנדס תוכנה, מנהל צוות עובדים",מפתח קוד לצ'יפים,nan,1.000000,1.000000,1.000000,6.000000,18.000000,55.000000,2000-11-10 00:00:00,7201,2512
8,9.000000,אשקלון,שופר סל,קופאית,nan,nan,קופאית,nan,nan,nan,1.000000,3.000000,2.000000,11.000000,52.000000,2022-11-11 00:00:00,4711,5230
9,10.000000,הרצליה,EMILIO,"הייטק, פיתוח תוכנה",nan,nan,מתכנת,nan,nan,nan,1.000000,1.000000,5.000000,16.000000,24.000000,2021-12-14 00:00:00,6201,2514


### Field-by-field investigation


### Analysis of ID column

The metadata defines the values within the column ID as a values of a running index.
The metadata defines column ID to be an Integer type.

What exactly does the running index identify, the respondent, the questionnaire, a submission, or a record?

The assignment mantions explicitly an framework that allaws resubmissions from the same participant within the the same servey timeline, which means the system must have a key per a single person within a single survey, and the only one can provide that in the data is an ID column.

In [33]:
# minus 1 column name record
len(data.ID) - 1

200

Conclussion: The data analysis confirms the 200 records of surveys

### Analysis of YeshuvAvoda column

Was the job location location was obtained throgh predifined categories or filled in as a free text?
The metadata states that it is stored in a text type field, but there is no mention of haw it was obtained.
From abserving the data it is clear that most likely come from dropdawn list or chousen category becouse the only iregulaarities in the column are bouth valid verssions of the same city, just one older verssion and the other is from newear national regestery verssion. 
Most likely bouth verssions still remained, so I would raise this issue with the relavent stakeholders and peers.

In [34]:
# missing values
data.YeshuvAvoda[(data.YeshuvAvoda.empty) or (data.YeshuvAvoda == None)]

Series([], Name: YeshuvAvoda, dtype: object)

Conclussion: The data analysis confirms that the fieald is a mandotory field which is also aligngs perfectly with my initial asamption.

### Analysis of ShemAvoda column

The metadata defines ShemAvoda as the name of the participant’s workplace.

How many records are empty, how many records are distinct relative to records that not empty, haw match records contain a Latin letters, and haw many records contain a mix of Hebrew and English letters in one cell?


In [35]:
shem_avoda = data.loc[data["ID"].notna(), "ShemAvoda"].astype("string").str.strip().replace("", pd.NA)

print("Empty records:", shem_avoda.isna().sum())
print("Distinct values among nonempty records:", shem_avoda.nunique(), "of", shem_avoda.count())

has_hebrew = shem_avoda.str.contains(r"[א-ת]", na=False)
has_latin = shem_avoda.str.contains(r"[A-Za-z]", na=False)
print("Records containing Hebrew letters:", has_hebrew.sum())
print("Records containing Latin letters:", has_latin.sum())
print("Records containing both Hebrew and Latin letters:", shem_avoda[has_hebrew & has_latin == True])

Empty records: 0
Distinct values among nonempty records: 82 of 200
Records containing Hebrew letters: 187
Records containing Latin letters: 14
Records containing both Hebrew and Latin letters: 139    יועצת IT עצמאית
Name: ShemAvoda, dtype: string


Conclusion: None of the 200 workplace records is empty. There are 82 distinct values among the 200 nonempty records (41%). Fourteen records contain Latin letters, and one of those also contains Hebrew letters in the same cell.

In [38]:
# Set the minimum cosine similarity a match must exceed.
ALPHA = 0.50

# Select each full ShemAvoda value containing at least one Latin letter.
latin_cells = shem_avoda.loc[has_latin]

# Collect the distinct Hebrew-only values already present in ShemAvoda.
hebrew_names = sorted(shem_avoda.loc[has_hebrew & ~has_latin].unique())

# Copy the survey ID and original workplace text for each selected row.
review = data.loc[latin_cells.index, ["ID", "ShemAvoda"]].copy()

# Encode the full selected values and the Hebrew-only values with LaBSE.
vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

# Compute cosine similarity between each selected value and every Hebrew-only value.
similarities = vectors[:len(latin_cells)] @ vectors[len(latin_cells):].T

# Find the highest-scoring Hebrew-only value for each selected row.
best_indices = similarities.argmax(axis=1)

# Record the highest cosine similarity for each selected row.
review["Cosine"] = similarities[np.arange(len(latin_cells)), best_indices]

# Align the highest-scoring Hebrew-only values with their survey rows.
best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=review.index, dtype="string")

# Replace the full workplace text when the score exceeds ALPHA and keep the original text otherwise.
review["Transformed"] = best_hebrew.where(review["Cosine"].gt(ALPHA), review["ShemAvoda"])

# Write the transformed text into ShemAvoda in a copy of the survey.
transformed_data = data.copy()
transformed_data.loc[review.index, "ShemAvoda"] = review["Transformed"]

# Display the original text, cosine scores, and transformed text.
display(review)

,ID,ShemAvoda,Cosine,Transformed
9,10.000000,EMILIO,0.428944,EMILIO
14,15.000000,NVIDIA,0.554522,נבידיה
37,38.000000,INTEL,0.752430,אינטל
46,47.000000,NVIDIA,0.554522,נבידיה
55,56.000000,COGNYTE,0.340869,COGNYTE
72,73.000000,EMILIO,0.428944,EMILIO
77,78.000000,NVIDIA,0.554522,נבידיה
100,101.000000,INTEL,0.752430,אינטל
103,104.000000,EMILIO,0.428944,EMILIO
109,110.000000,NVIDIA,0.554522,נבידיה


Which Hebrew workplace value does LaBSE rank first for each standalone name written in Latin letters? Adjust `alpha` below to set the minimum cosine similarity a candidate must exceed to be reviewed.


## Preparation of the remaining fields

The following investigation continues from `transformed_data`. For each field, the metadata establishes its meaning, the displayed records test the preparation decision, and the next cell applies that decision. `prepared_data` contains the working values. Original answers remain in `data`, and `preparation_audit` records missing, withheld, unresolved, and contradictory responses. No occupation classifier is trained in this notebook section.

In [8]:
# Verify that the workbook is the source used in the preceding analysis.
assert hashlib.sha256(raw).hexdigest() == SHA
# Continue with identified survey records after the workplace transformation.
survey = transformed_data.loc[transformed_data["ID"].notna()].copy()
assert survey["ID"].is_unique
# Keep the source record index while preparing a separate table.
prepared_data = survey.copy()
preparation_audit = survey[["ID"]].copy()
# Read the metadata's code definitions in addition to its three descriptive columns.
metadata_definitions = pd.read_excel(io.BytesIO(raw), sheet_name="Metadata", usecols="A:D", nrows=len(metadata))
# Keep text fields separate and treat whitespace-only answers as missing.
text_fields = ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]
text_values = survey[text_fields].astype("string").apply(lambda column: column.str.strip().replace("", pd.NA))
# Record response states without treating them as occupational descriptions.
response_states = pd.DataFrame("Observed", index=survey.index, columns=text_fields)
response_states = response_states.mask(text_values.isna(), "Missing").mask(text_values.eq('כנ"ל').fillna(False), "Reference").mask(text_values.isin(["חסוי", "לא יכול לפרט"]), "Withheld")
display(pd.DataFrame({"Populated": text_values.count(), "Empty": text_values.isna().sum(), "Distinct": text_values.nunique()}))


,Populated,Empty,Distinct
ShemAvoda,200,0,78
SugAvoda,175,25,78
ShemMachlaka,136,64,61
SugMachlaka,66,134,35
EzoAvoda,193,7,84
TeurPeula,64,136,32
TeurTafkid,28,172,16


### Analysis of SugAvoda column

The metadata defines `SugAvoda` as the workplace's main activity. Does each answer describe the organization, or does it repeat the participant's occupation? Which missing or withheld answers require a different representation from substantive activity text?

In [9]:
# Count response states and inspect exact repetitions of the work description.
display(response_states["SugAvoda"].value_counts().rename("Records"))
repeated_activity = text_values["SugAvoda"].eq(text_values["EzoAvoda"]).fillna(False)
display(survey.loc[repeated_activity, ["ID", "ShemAvoda", "SugAvoda", "EzoAvoda"]])
# Inspect full activity values containing Latin letters alongside the participant's work.
display(survey.loc[text_values["SugAvoda"].str.contains(r"[A-Za-z]", na=False), ["ID", "ShemAvoda", "SugAvoda", "EzoAvoda"]])


SugAvoda
Observed    171
Missing      25
Withheld      4
Name: Records, dtype: int64

,ID,ShemAvoda,SugAvoda,EzoAvoda
5,6.000000,משטרה,שוטר,שוטר
8,9.000000,שופר סל,קופאית,קופאית
36,37.000000,משטרה,שוטרת,שוטרת
49,50.000000,משרד ממשלתי,חסוי,חסוי
53,54.000000,אוניברסיטת תל אביב,חוקר במעבדה,חוקר במעבדה
68,69.000000,משטרה,שוטר,שוטר
71,72.000000,שופר סל,קופאית,קופאית
99,100.000000,משטרה,שוטרת,שוטרת
102,103.000000,שופר סל,קופאית,קופאית
112,113.000000,משרד ממשלתי,חסוי,חסוי


,ID,ShemAvoda,SugAvoda,EzoAvoda
2,3.000000,קריית הממשלה,מתן שירות למשתמשים באתר GOV,נותנת תמיכה ללקוחות שמבקשים עזרה
65,66.000000,בזק,מתן שירות למשתמשים באתר GOV,נותנת תמיכה ללקוחות שמבקשים עזרה
139,140.000000,קוסמטיקאית עצמאית,יועצת IT,יועצת IT


**Conclusion and preparation:** The field has 175 populated answers and 25 blanks. Twenty-seven answers exactly repeat `EzoAvoda`, including two withheld answers. Four answers are `חסוי`. The field therefore contains both organizational context and descriptions of individual work. Keep it separate from the occupation fields, retain its original wording, and remove withheld markers from the text vocabulary while recording their state. English terms remain intact because company-name matching does not establish equivalent activity descriptions.

In [10]:
# Retain substantive workplace activity text and preserve the reason for an empty model input.
prepared_data["SugAvoda"] = text_values["SugAvoda"].where(response_states["SugAvoda"].eq("Observed"), "").fillna("")
preparation_audit["SugAvoda_state"] = response_states["SugAvoda"]
# Verify the prepared field against its response states.
display(pd.crosstab(preparation_audit["SugAvoda_state"], prepared_data["SugAvoda"].eq(""), colnames=["Prepared text empty"]))


Prepared text empty,False,True
SugAvoda_state,,
Missing,0,25
Observed,171,0
Withheld,0,4


### Analysis of ShemMachlaka column

The metadata asks for the department or division. Does a missing department name imply that there is no department, and can Latin abbreviations be replaced safely from the available information?

In [11]:
# Compare missing department names with missing department activities.
display(pd.crosstab(text_values["ShemMachlaka"].isna(), text_values["SugMachlaka"].isna(), rownames=["Department name missing"], colnames=["Department activity missing"]))
# Inspect the abbreviations in the full workplace and occupational context.
display(survey.loc[text_values["ShemMachlaka"].str.contains(r"[A-Za-z]", na=False), ["ID", "ShemAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula"]])
display(response_states["ShemMachlaka"].value_counts().rename("Records"))


Department activity missing,False,True
Department name missing,,
False,66,70
True,0,64


,ID,ShemAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula
40,41.000000,מיקרוסופט,R&D,R&D,מתכנת,nan
55,56.000000,COGNYTE,R&D,nan,DATA SCIENTIST,בונה מודלים של AI
118,119.000000,COGNYTE,R&D,nan,DATA ANALYST,nan
136,137.000000,אמדוקס,DELIVERY,nan,"ר""ש פיתוח",פיתוח תכנה
138,139.000000,אמדוקס,HR,משאבי אנוש,מנהלת משאבי אנוש,מנהלת משאבי אנוש של עובדיםן בישראל
143,144.000000,אמדוקס,IT,IT,ראש צוות,תמיכה במחשוב


ShemMachlaka
Observed    132
Missing      64
Withheld      4
Name: Records, dtype: int64

**Conclusion and preparation:** There are 136 populated department names and 64 blanks. Every blank department name also has a blank department activity, while 70 populated department names have no activity answer. This establishes absent responses, not absent departments. Keep department text as its own feature. Preserve `R&D`, `HR`, `IT`, and `DELIVERY` as recorded because expanding an abbreviation is a semantic transformation that the workbook does not define. Store missing and withheld states separately.

In [12]:
# Keep department descriptions and abbreviations without assigning an occupation to the department.
prepared_data["ShemMachlaka"] = text_values["ShemMachlaka"].where(response_states["ShemMachlaka"].eq("Observed"), "").fillna("")
preparation_audit["ShemMachlaka_state"] = response_states["ShemMachlaka"]
display(preparation_audit["ShemMachlaka_state"].value_counts().rename("Records"))


ShemMachlaka_state
Observed    132
Missing      64
Withheld      4
Name: Records, dtype: int64

### Analysis of SugMachlaka column

The department activity can explain a name such as “microeconomics” or “customer support.” When the answer is `כנ"ל`, does the preceding context identify one unambiguous activity to copy?

In [13]:
# Display every reference together with both possible antecedents and the work description.
department_references = response_states["SugMachlaka"].eq("Reference")
display(survey.loc[department_references | survey["ID"].eq(1), ["ID", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda"]])
display(response_states["SugMachlaka"].value_counts().rename("Records"))


,ID,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda
0,1.000000,"איסוף, עיבוד ופרסום נתונים",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל אגף
2,3.000000,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה
33,34.000000,תיקון קווי מים,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה
65,66.000000,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה
96,97.000000,הספקת מים,הנדסה,"כנ""ל",תכנון צינורות
128,129.000000,קבלת קהל,תמיכה בלקוחות,"כנ""ל",קבלת קהל
164,165.000000,קבלת קהל,תמיכה בלקוחות,"כנ""ל",קבלת קהל
186,187.000000,תיקון קווי חשמל,תמיכה בתקלות,"כנ""ל",נותנת תמיכה בהפשסקות חשמל
192,193.000000,קבלת קהל,תמיכה בלקוחות,"כנ""ל",nan


SugMachlaka
Missing      134
Observed      58
Reference      8
Name: Records, dtype: int64

**Conclusion and preparation:** The field has 66 populated answers, including eight references, and 134 blanks. The references may point to the department name or an earlier description, and the workbook records no explicit link. Automatically copying the employer's activity would impose a relationship that is not established. Preserve substantive department activities, replace the reference token with an empty text input, and retain an `Unresolved reference` state. The earlier fields remain available independently.

In [14]:
# Keep unresolved references distinct from missing department activities.
prepared_data["SugMachlaka"] = text_values["SugMachlaka"].where(response_states["SugMachlaka"].eq("Observed"), "").fillna("")
preparation_audit["SugMachlaka_state"] = response_states["SugMachlaka"].replace({"Reference": "Unresolved reference"})
display(preparation_audit["SugMachlaka_state"].value_counts().rename("Records"))


SugMachlaka_state
Missing                 134
Observed                 58
Unresolved reference      8
Name: Records, dtype: int64

### Analysis of EzoAvoda column

The metadata defines this field as the work performed by the participant. Can the task description or title supply occupational evidence when this main description is missing, and which answers provide no occupational content?

In [15]:
# Inspect every missing main work description with the two complementary occupation fields.
role_fields = ["EzoAvoda", "TeurPeula", "TeurTafkid"]
display(survey.loc[text_values["EzoAvoda"].isna(), ["ID", "ShemAvoda"] + role_fields + ["SemelMishlachSofi"]])
# Inspect withheld answers and full bilingual descriptions without extracting words.
display(survey.loc[response_states["EzoAvoda"].eq("Withheld") | text_values["EzoAvoda"].str.contains(r"[A-Za-z]", na=False), ["ID"] + role_fields])
display(response_states["EzoAvoda"].value_counts().rename("Records"))


,ID,ShemAvoda,EzoAvoda,TeurPeula,TeurTafkid,SemelMishlachSofi
44,45.000000,מסעדה,nan,nan,nan,XXXX
107,108.000000,מסעדה,nan,nan,nan,XXXX
141,142.000000,שופרסל,nan,nan,nan,XXXX
163,164.000000,סומך חייקין,nan,nan,nan,2411
165,166.000000,יועצץ מחשוב,nan,nan,nan,2511
170,171.000000,בנק הפועלים,nan,nan,מנהלת משאבי אנוש,2423
192,193.000000,דואר ישראל,nan,nan,nan,XXXX


,ID,EzoAvoda,TeurPeula,TeurTafkid
17,18.000000,לא יכול לפרט,nan,nan
23,24.000000,DATA ANALYST,nan,nan
49,50.000000,חסוי,nan,nan
55,56.000000,DATA SCIENTIST,בונה מודלים של AI,בונה מודלים של AI
80,81.000000,לא יכול לפרט,nan,nan
86,87.000000,DATA ANALYST,nan,nan
112,113.000000,חסוי,nan,nan
118,119.000000,DATA ANALYST,nan,nan
139,140.000000,יועצת IT,יועצת IT,יועצת IT
144,145.000000,DATA ANALYST,nan,nan


EzoAvoda
Observed    189
Missing       7
Withheld      4
Name: Records, dtype: int64

**Conclusion and preparation:** Seven main work descriptions are blank. Record 171 still has an HR title, while six records have none of the three occupational descriptions. Four other work descriptions withhold the information. Keep all three occupation fields separate so that available tasks or titles can contribute without being presented as the participant's missing answer. Retain bilingual work descriptions, remove withholding phrases from the vocabulary, and preserve the missing or withheld state.

In [16]:
# Prepare direct work descriptions while preserving the distinction between missing and withheld answers.
prepared_data["EzoAvoda"] = text_values["EzoAvoda"].where(response_states["EzoAvoda"].eq("Observed"), "").fillna("")
preparation_audit["EzoAvoda_state"] = response_states["EzoAvoda"]
display(preparation_audit["EzoAvoda_state"].value_counts().rename("Records"))


EzoAvoda_state
Observed    189
Missing       7
Withheld      4
Name: Records, dtype: int64

### Analysis of TeurPeula column

Do the main tasks add information beyond the work description? For `כנ"ל`, is the immediately preceding work description a usable occupational antecedent?

In [17]:
# Inspect tasks that add specific duties, including a work-description disagreement.
display(survey.loc[survey["ID"].isin([3, 7, 34, 38, 56, 140]), ["ID"] + role_fields])
# Examine every task reference before choosing its replacement.
task_references = response_states["TeurPeula"].eq("Reference")
display(survey.loc[task_references, ["ID"] + role_fields])
display(response_states["TeurPeula"].value_counts().rename("Records"))


,ID,EzoAvoda,TeurPeula,TeurTafkid
2,3.000000,נותנת תמיכה ללקוחות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,nan
6,7.000000,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,nan
33,34.000000,נותנת תמיכה ללקוחות שמבקשים עזרה,עוזר לתקן צינורות,nan
37,38.000000,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,nan
55,56.000000,DATA SCIENTIST,בונה מודלים של AI,בונה מודלים של AI
139,140.000000,יועצת IT,יועצת IT,יועצת IT


,ID,EzoAvoda,TeurPeula,TeurTafkid
0,1.000000,סגן מנהל אגף,"כנ""ל",סגן מנהל אגף
1,2.000000,טכנאית רדיותרפיה,"כנ""ל","כנ""ל"
11,12.000000,"טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות","כנ""ל",כלכלן
31,32.000000,סגן מנהל אגף,"כנ""ל",סגן מנהל אגף
32,33.000000,טכנאית רדיותרפיה,"כנ""ל","כנ""ל"
42,43.000000,"טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות","כנ""ל",כלכלן
63,64.000000,מנהל אגף,"כנ""ל",סגן מנהל אגף
64,65.000000,טכנאית רדיותרפיה,"כנ""ל","כנ""ל"
74,75.000000,"טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות","כנ""ל",כלכלן
94,95.000000,סוקר,"כנ""ל",סוקר


TeurPeula
Missing      136
Observed      52
Reference     12
Name: Records, dtype: int64

**Conclusion and preparation:** There are 64 populated task answers, including 12 references, and 136 blanks. Record 3 specifies online government-service support, and record 7 distinguishes production supervision from the general phrase “clean-room worker.” All 12 references follow a substantive work description. Resolve these references to that description as an explicit questionnaire-order interpretation and record the source field. Preserve distinct task wording, including disagreements, instead of overwriting it with the title or employer activity.

In [18]:
# Copy the immediately preceding work description only for the inspected task references.
resolved_tasks = task_references & prepared_data["EzoAvoda"].ne("")
prepared_data["TeurPeula"] = text_values["TeurPeula"].where(response_states["TeurPeula"].eq("Observed"), "").fillna("")
prepared_data.loc[resolved_tasks, "TeurPeula"] = prepared_data.loc[resolved_tasks, "EzoAvoda"]
preparation_audit["TeurPeula_state"] = response_states["TeurPeula"].replace({"Reference": "Unresolved reference"})
preparation_audit.loc[resolved_tasks, "TeurPeula_state"] = "Resolved from EzoAvoda"
display(pd.DataFrame({"ID": survey.loc[task_references, "ID"], "Original": survey.loc[task_references, "TeurPeula"], "Prepared": prepared_data.loc[task_references, "TeurPeula"]}))


,ID,Original,Prepared
0,1.000000,"כנ""ל",סגן מנהל אגף
1,2.000000,"כנ""ל",טכנאית רדיותרפיה
11,12.000000,"כנ""ל","טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות"
31,32.000000,"כנ""ל",סגן מנהל אגף
32,33.000000,"כנ""ל",טכנאית רדיותרפיה
42,43.000000,"כנ""ל","טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות"
63,64.000000,"כנ""ל",מנהל אגף
64,65.000000,"כנ""ל",טכנאית רדיותרפיה
74,75.000000,"כנ""ל","טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות"
94,95.000000,"כנ""ל",סוקר


### Analysis of TeurTafkid column

The title is requested only when one exists. Which populated values are titles, references, or placeholders, and do any titles disagree with the work description?

In [19]:
# Inspect references, the zero placeholder, repeated task descriptions, and the manager/deputy disagreement.
title_references = response_states["TeurTafkid"].eq("Reference")
title_placeholder = text_values["TeurTafkid"].isin(["0", "0.0"])
repeated_title = text_values["TeurTafkid"].eq(text_values["TeurPeula"]).fillna(False)
display(survey.loc[title_references | title_placeholder | repeated_title | survey["ID"].eq(64), ["ID"] + role_fields + ["MenahelEtMi"]])
print("Missing titles:", int(text_values["TeurTafkid"].isna().sum()))


Missing titles: 172


,ID,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi
1,2.000000,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan
32,33.000000,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan
38,39.000000,"מהנדס תוכנה, מנהל צוות פיתוח",פיתוח קוד,0,2.000000
55,56.000000,DATA SCIENTIST,בונה מודלים של AI,בונה מודלים של AI,2.000000
63,64.000000,מנהל אגף,"כנ""ל",סגן מנהל אגף,3.000000
64,65.000000,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan
95,96.000000,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan
136,137.000000,"ר""ש פיתוח",פיתוח תכנה,פיתוח תכנה,nan
137,138.000000,מנהל פיתוח,מנהל פיתןח של 2 מוצרים,מנהל פיתןח של 2 מוצרים,2.000000
138,139.000000,מנהלת משאבי אנוש,מנהלת משאבי אנוש של עובדיםן בישראל,מנהלת משאבי אנוש של עובדיםן בישראל,3.000000


**Conclusion and preparation:** There are 28 populated titles and 172 blanks. Four references form the same radiotherapy chain through tasks to the work description. Record 39 contains `0`, which is a placeholder rather than a title. Six substantive titles repeat task text. Record 64 says department manager in the work description and deputy department manager in the title. Resolve the four inspected references, remove the zero placeholder, and keep the conflicting answers in separate fields with a review flag. Missing titles are legitimate optional responses and should not be filled with an invented title.

In [20]:
# Resolve title references only when the preceding prepared task text equals the work description.
resolved_titles = title_references & prepared_data["TeurPeula"].ne("") & prepared_data["TeurPeula"].eq(prepared_data["EzoAvoda"])
prepared_data["TeurTafkid"] = text_values["TeurTafkid"].where(response_states["TeurTafkid"].eq("Observed") & ~title_placeholder, "").fillna("")
prepared_data.loc[resolved_titles, "TeurTafkid"] = prepared_data.loc[resolved_titles, "TeurPeula"]
preparation_audit["TeurTafkid_state"] = response_states["TeurTafkid"].replace({"Reference": "Unresolved reference"})
preparation_audit.loc[resolved_titles, "TeurTafkid_state"] = "Resolved from TeurPeula"
preparation_audit.loc[title_placeholder, "TeurTafkid_state"] = "Placeholder"
# Retain the observed contradiction for review without selecting either answer as truth.
preparation_audit["Title_work_disagreement"] = survey["ID"].eq(64)
display(pd.DataFrame({"ID": survey.loc[title_references | title_placeholder, "ID"], "Original": survey.loc[title_references | title_placeholder, "TeurTafkid"], "Prepared": prepared_data.loc[title_references | title_placeholder, "TeurTafkid"]}))


,ID,Original,Prepared
1,2.000000,"כנ""ל",טכנאית רדיותרפיה
32,33.000000,"כנ""ל",טכנאית רדיותרפיה
38,39.000000,0,
64,65.000000,"כנ""ל",טכנאית רדיותרפיה
95,96.000000,"כנ""ל",טכנאית רדיותרפיה


### Analysis of MenahelEtMi column

Do codes 1, 2, and 3 identify the same kind of numerical quantity as age, and does a blank answer establish that the participant manages nobody?

In [21]:
# Read the management definitions and the values occurring in identified records.
display(metadata_definitions.loc[metadata_definitions["משתנה"].eq("MenahelEtMi")])
display(survey["MenahelEtMi"].value_counts(dropna=False).sort_index().rename("Records"))
# Separate the non-record worksheet calculation from survey responses.
display(data.loc[data["ID"].isna(), ["ID", "MenahelEtMi"]])
# Examine a managerial abbreviation accompanied by a missing management answer.
display(survey.loc[survey["ID"].eq(137), ["ID"] + role_fields + ["MenahelEtMi"]])


,משתנה,הסבר,פורמט,ערכים אפשריים
9,MenahelEtMi,האם הנסקר מנהל או ממונה על עובדים אחרים?,INTEGER,1. מנהל עובדים 2. מנהל מנהלים 3. מנהל עובדים ומנהלים


MenahelEtMi
1.0      8
2.0     11
3.0     13
NaN    168
Name: Records, dtype: int64

,ID,MenahelEtMi
200,nan,1.725000


,ID,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi
136,137.000000,"ר""ש פיתוח",פיתוח תכנה,פיתוח תכנה,nan


**Conclusion and preparation:** Thirty-two records contain a management code and 168 are blank. The value 1.725 belongs to a worksheet row without a survey ID and is already excluded from `survey`. Codes 1, 2, and 3 identify whom the participant manages. Record 137 contains a managerial abbreviation but no answer, so missing cannot be established as no management. Represent missing by 0 while retaining a missing-state flag. Treat the four resulting values as categories in the feature encoder so that numerical distances do not impose an unsupported management scale.

In [22]:
# Validate the observed codes before representing missing responses by a distinct zero code.
assert survey["MenahelEtMi"].dropna().isin([1, 2, 3]).all()
preparation_audit["MenahelEtMi_missing"] = survey["MenahelEtMi"].isna()
prepared_data["MenahelEtMi"] = survey["MenahelEtMi"].fillna(0).astype("Int64").astype("string")
display(prepared_data["MenahelEtMi"].value_counts().sort_index().rename("Prepared records"))


MenahelEtMi
0    168
1      8
2     11
3     13
Name: Prepared records, dtype: Int64

### Analysis of MaamadAvoda column

Do employment-status numbers define an ordered scale, and how should an unanswered status affect the salary-source eligibility check?

In [23]:
# Read all employment-status definitions and count the codes that occur.
display(metadata_definitions.loc[metadata_definitions["משתנה"].eq("MaamadAvoda")])
display(survey["MaamadAvoda"].value_counts(dropna=False).sort_index().rename("Records"))
display(survey.loc[survey["MaamadAvoda"].isna(), ["ID", "MaamadAvoda", "MakorSachar", "EzoAvoda"]])


,משתנה,הסבר,פורמט,ערכים אפשריים
10,MaamadAvoda,מעמד הנסקר במקום עבודה זה,INTEGER,"1. שכיר (כולל שכיר בעל חברת בע""מ) 2. עצמאי שמעסיק 1-2 עובדים 3. עצמאי שמעסיק 3 עובדים ויותר 4. עצמאי שלא מעסיק עובדים 5. חבר קואופרטיב 6. חבר קיבוץ 7. בן משפחה העובד ללא תשלום"


MaamadAvoda
1.0    177
2.0      1
4.0     10
6.0     10
NaN      2
Name: Records, dtype: int64

,ID,MaamadAvoda,MakorSachar,EzoAvoda
44,45.000000,nan,nan,nan
107,108.000000,nan,nan,nan


**Conclusion and preparation:** The observed statuses are 1, 2, 4, and 6, with two missing answers. The seven metadata definitions describe different employment arrangements rather than a common increasing quantity. Prepare a categorical field for one-hot encoding, retain missing as its own category, and use the original numeric codes when evaluating eligibility. Do not infer an employment arrangement from the employer name.

In [24]:
# Retain nominal employment codes and an explicit missing category.
assert survey["MaamadAvoda"].dropna().isin(range(1, 8)).all()
prepared_data["MaamadAvoda"] = survey["MaamadAvoda"].astype("Int64").astype("string").fillna("Missing")
display(prepared_data["MaamadAvoda"].value_counts().rename("Prepared records"))


MaamadAvoda
1          177
4           10
6           10
Missing      2
2            1
Name: Prepared records, dtype: Int64

### Analysis of MakorSachar column

The metadata restricts this question to employment codes 1 and 5. Which blanks are structurally inapplicable, which have unknown eligibility, and which populated answers contradict that condition?

In [25]:
# Cross-check eligibility against every salary-source response.
display(metadata_definitions.loc[metadata_definitions["משתנה"].eq("MakorSachar")])
employment_status = survey["MaamadAvoda"]
salary_source = survey["MakorSachar"]
salary_eligible = employment_status.isin([1, 5])
salary_conflict = employment_status.notna() & ~salary_eligible & salary_source.notna()
display(pd.crosstab(employment_status.astype("Int64").astype("string").fillna("Missing"), salary_source.astype("Int64").astype("string").fillna("Missing")))
display(survey.loc[salary_conflict, ["ID", "ShemAvoda", "MaamadAvoda", "MakorSachar"]])


,משתנה,הסבר,פורמט,ערכים אפשריים
11,MakorSachar,מקור השכר של הנסקר (השדה מלא רק במידה והנסקר שכיר או חבר קואופרטיב - כפי שצויין בשדה :MaamadAvoda ),INTEGER,"1. מקום העבודה 2. מחברת קבלן: בית תוכנה, חברת שמירה, ניקיון וכדומה 3. מחברת כוח אדם 4. אחר"


MakorSachar,1,2,3,4,Missing
MaamadAvoda,,,,,
1,136,26,11,4,0
2,1,0,0,0,0
4,7,0,0,1,2
6,0,0,0,0,10
Missing,0,0,0,0,2


,ID,ShemAvoda,MaamadAvoda,MakorSachar
33,34.000000,מקורות,4.000000,4.000000
48,49.000000,קוסמטיקאית עצמאית,4.000000,1.000000
111,112.000000,קוסמטיקאית עצמאית,4.000000,1.000000
139,140.000000,קוסמטיקאית עצמאית,4.000000,1.000000
165,166.000000,יועצץ מחשוב,4.000000,1.000000
175,176.000000,קוסמטיקאית עצמאית,4.000000,1.000000
186,187.000000,"חח""י",4.000000,1.000000
189,190.000000,הרודס,2.000000,1.000000
197,198.000000,קוסמטיקאית,4.000000,1.000000


**Conclusion and preparation:** All 177 eligible records have a salary-source answer. Nine ineligible records also provide one, 12 ineligible records leave it blank, and two records have unknown employment status and no salary source. Prepare distinct categories for `Conflict`, `Not applicable`, `Unknown eligibility`, and a missing eligible answer. Preserve valid source codes as nominal categories. The original conflicting answers remain available for correction, but their numbers do not enter the model as valid salary-source answers.

In [26]:
# Validate salary codes and separate questionnaire states before categorical encoding.
assert salary_source.dropna().isin([1, 2, 3, 4]).all()
prepared_data["MakorSachar"] = salary_source.astype("Int64").astype("string").fillna("Missing eligible answer")
prepared_data.loc[employment_status.isna(), "MakorSachar"] = "Unknown eligibility"
prepared_data.loc[employment_status.notna() & ~salary_eligible, "MakorSachar"] = "Not applicable"
prepared_data.loc[salary_conflict, "MakorSachar"] = "Conflict"
preparation_audit["Salary_eligibility_conflict"] = salary_conflict
display(prepared_data["MakorSachar"].value_counts().rename("Prepared records"))


MakorSachar
1                      136
2                       26
Not applicable          12
3                       11
Conflict                 9
4                        4
Unknown eligibility      2
Name: Prepared records, dtype: Int64

### Analysis of TeudaGvoha column

Does a larger qualification code consistently represent higher education, and can years of schooling resolve apparent inconsistencies without additional information?

In [27]:
# Inspect the complete code definitions, including other and no certificate.
display(metadata_definitions.loc[metadata_definitions["משתנה"].eq("TeudaGvoha")])
display(survey["TeudaGvoha"].value_counts(dropna=False).sort_index().rename("Records"))
# Cross-check qualifications with years of schooling without substituting either answer.
display(pd.crosstab(survey["TeudaGvoha"], survey["shnotlimud"]))
display(survey.loc[survey["shnotlimud"].eq(7), ["ID", "TeudaGvoha", "shnotlimud", "Gil"]])


,משתנה,הסבר,פורמט,ערכים אפשריים
12,TeudaGvoha,התעודה או התואר הגבוה ביותר של הנסקר,INTEGER,"1. תעוד סיום של בי""ס יסודי או חטיבת ביניים 2. תעודת סיום תיכון שאינה תעודת בגרות 3. תעודת בגרות 4. תעודת סיום של בי""ס על תיכוני שאינה תעודה אקדמית 5. תואר אקדמי ראשון B.A, או תואר מקביל 6. תואר אקדמי שני M.A, או תואר מקביל כולל דוקטור לרפואה 7. תואר אקדמי שלישי Ph.D, או תואר מקביל 8. תעודה אחרת, ציין 9. לא קיבל אף תעודה"


TeudaGvoha
1.0     3
2.0    12
3.0    57
4.0    58
5.0    25
6.0    39
7.0     4
NaN     2
Name: Records, dtype: int64

shnotlimud,7.000000,8.000000,10.000000,11.000000,12.000000,13.000000,14.000000,15.000000,16.000000,17.000000,18.000000,20.000000,22.000000
TeudaGvoha,,,,,,,,,,,,,
1.000000,0,0,0,0,3,0,0,0,0,0,0,0,0
2.000000,0,1,6,5,0,0,0,0,0,0,0,0,0
3.000000,2,0,0,0,45,6,4,0,0,0,0,0,0
4.000000,0,0,0,0,5,19,20,14,0,0,0,0,0
5.000000,0,0,0,0,0,0,0,16,9,0,0,0,0
6.000000,0,0,0,0,0,0,0,0,2,10,16,3,8
7.000000,0,0,0,0,0,0,0,0,0,0,0,4,0


,ID,TeudaGvoha,shnotlimud,Gil
33,34.000000,3.000000,7.000000,29.000000
192,193.000000,3.000000,7.000000,59.000000


**Conclusion and preparation:** Codes 8 and 9 mean another certificate and no certificate, so the entire coding scheme is not an increasing education scale. Two qualification answers are missing. Records 34 and 193 combine matriculation with seven years of schooling, which merits review but does not identify which answer is wrong. Use one-hot encoding with an explicit missing category, retain both reported values, and flag the observed education inconsistency rather than inventing a replacement.

In [28]:
# Prepare the nominal qualification categories and retain the observed cross-field issue.
assert survey["TeudaGvoha"].dropna().isin(range(1, 10)).all()
prepared_data["TeudaGvoha"] = survey["TeudaGvoha"].astype("Int64").astype("string").fillna("Missing")
preparation_audit["Qualification_schooling_review"] = survey["TeudaGvoha"].eq(3) & survey["shnotlimud"].eq(7)
display(prepared_data["TeudaGvoha"].value_counts().sort_index().rename("Prepared records"))


TeudaGvoha
1           3
2          12
3          57
4          58
5          25
6          39
7           4
Missing     2
Name: Prepared records, dtype: Int64

### Analysis of shnotlimud column

The field measures years of schooling. Are values missing, fractional, negative, or greater than the participant's age, and does the metadata justify a minimum of eight years?

In [29]:
# Examine the range, integer format, and cross-field plausibility of schooling.
school_years = pd.to_numeric(survey["shnotlimud"], errors="coerce")
invalid_schooling = school_years.notna() & (school_years.lt(0) | school_years.mod(1).ne(0))
schooling_exceeds_age = school_years.gt(survey["Gil"])
display(school_years.agg(["count", "min", "max", "median", "nunique"]))
display(survey.loc[school_years.isna() | invalid_schooling | schooling_exceeds_age | school_years.eq(7), ["ID", "shnotlimud", "TeudaGvoha", "Gil"]])


count      198.0
min          7.0
max         22.0
median      14.0
nunique     13.0
Name: shnotlimud, dtype: float64

,ID,shnotlimud,TeudaGvoha,Gil
33,34.000000,7.000000,3.000000,29.000000
44,45.000000,nan,nan,nan
107,108.000000,nan,nan,nan
192,193.000000,7.000000,3.000000,59.000000


**Conclusion and preparation:** The 198 observed values are whole numbers from 7 to 22, with two missing answers. No schooling value is negative, fractional, or greater than the recorded age. The metadata specifies INTEGER but no minimum of eight, so the two seven-year entries remain unchanged. Retain schooling as numeric, preserve a missing indicator, and fit median imputation and scaling on training rows only.

In [30]:
# Keep valid numeric measurements and mark invalid inputs as missing if encountered.
prepared_data["shnotlimud"] = school_years.mask(invalid_schooling)
preparation_audit["shnotlimud_missing"] = prepared_data["shnotlimud"].isna()
preparation_audit["shnotlimud_invalid"] = invalid_schooling
preparation_audit["Schooling_exceeds_age"] = schooling_exceeds_age
display(prepared_data["shnotlimud"].agg(["count", "min", "max", "median"]))


count     198.0
min         7.0
max        22.0
median     14.0
Name: shnotlimud, dtype: float64

### Analysis of Gil column

Does the workbook contain the participant's age at the survey or enough information to calculate a current age? Which recorded ages fail the declared integer format?

In [31]:
# Inspect the observed age range and all missing or invalid values.
age = pd.to_numeric(survey["Gil"], errors="coerce")
invalid_age = age.notna() & (age.lt(0) | age.mod(1).ne(0))
display(age.agg(["count", "min", "max", "median", "nunique"]))
display(survey.loc[age.isna() | invalid_age, ["ID", "Gil", "shnotlimud", "TeudaGvoha", "MaamadAvoda"]])


count      198.0
min         22.0
max         78.0
median      37.0
nunique     29.0
Name: Gil, dtype: float64

,ID,Gil,shnotlimud,TeudaGvoha,MaamadAvoda
44,45.000000,nan,nan,nan,nan
107,108.000000,nan,nan,nan,nan


**Conclusion and preparation:** There are 198 observed ages, all whole numbers from 22 to 78. Records 45 and 108 are missing age and the other displayed background fields. The workbook contains no date of birth or individual survey date, so there is no justified current-age recalculation. Retain the reported numeric age, preserve missingness, and estimate its imputation value and scaling parameters from training rows only.

In [32]:
# Preserve the age reported in the survey and its missing-state information.
prepared_data["Gil"] = age.mask(invalid_age)
preparation_audit["Gil_missing"] = prepared_data["Gil"].isna()
preparation_audit["Gil_invalid"] = invalid_age
display(prepared_data["Gil"].agg(["count", "min", "max", "median"]))


count     198.0
min        22.0
max        78.0
median     37.0
Name: Gil, dtype: float64

### Analysis of TarichHathala column

The metadata defines the start of the participant's role. Can the recorded dates be parsed, and is there a survey reference date that would make a tenure calculation meaningful?

In [33]:
# Parse role-start dates and inspect populated values that fail parsing.
start_dates = pd.to_datetime(survey["TarichHathala"], errors="coerce")
display(start_dates.agg(["count", "min", "max", "nunique"]))
display(survey.loc[survey["TarichHathala"].notna() & start_dates.isna(), ["ID", "TarichHathala"]])


count                      200
min        1999-06-06 00:00:00
max        2026-06-06 00:00:00
nunique                     42
Name: TarichHathala, dtype: object

,ID,TarichHathala


**Conclusion and preparation:** All 200 dates parse successfully, with 42 distinct dates between 6 June 1999 and 6 June 2026. A start date alone is not tenure, and the workbook supplies no survey reference date. Store the validated dates for traceability and omit this field from the primary feature matrix. Do not subtract them from today's date or fabricate a survey date.

In [34]:
# Preserve validated dates while explicitly excluding them from primary predictors.
prepared_data["TarichHathala"] = start_dates
preparation_audit["TarichHathala_invalid"] = survey["TarichHathala"].notna() & start_dates.isna()
excluded_predictors = ["ID", "TarichHathala", "SemelMishlachSofi", "SemelAnafSofi"]
display(prepared_data[["ID", "TarichHathala"]].head())


,ID,TarichHathala
0,1.000000,2020-06-15 00:00:00
1,2.000000,2022-07-01 00:00:00
2,3.000000,2021-06-06 00:00:00
3,4.000000,2024-07-01 00:00:00
4,5.000000,2010-01-11 00:00:00


### Analysis of SemelAnafSofi column

The metadata defines a four-position industry code. Are short numeric values display-format artifacts, and should a partial industry code be confused with an incomplete occupation target?

In [35]:
# Compare stored values with their Excel number formats before restoring leading zeros.
from openpyxl import load_workbook
source_workbook = load_workbook(io.BytesIO(raw), read_only=True, data_only=True)
source_sheet = source_workbook["Survey Dataset"]
industry_position = list(next(source_sheet.values)).index("SemelAnafSofi") + 1
industry_storage = pd.DataFrame({"ID": survey["ID"], "Stored": survey["SemelAnafSofi"], "Excel format": [source_sheet.cell(int(index) + 2, industry_position).number_format for index in survey.index]})
industry_codes = survey["SemelAnafSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
display(industry_storage.loc[industry_codes.str.fullmatch(r"[0-9]{1,3}", na=False)])
display(survey.loc[industry_codes.str.contains("X", na=False), ["ID", "SemelAnafSofi", "SemelMishlachSofi"]])
source_workbook.close()


,ID,Stored,Excel format
29,30.000000,123,0000
61,62.000000,123,0000
92,93.000000,123,0000
124,125.000000,123,0000
150,151.000000,124,0000


,ID,SemelAnafSofi,SemelMishlachSofi
4,5.000000,XXXX,5322
35,36.000000,XXXX,5322
67,68.000000,XXXX,5322
98,99.000000,XXXX,5322
127,128.000000,XXXX,5322
152,153.000000,851X,1345
153,154.000000,XXXX,5322
173,174.000000,852X,2330
182,183.000000,3XXX,8183


**Conclusion and preparation:** Five stored industry values need a leading zero to reproduce Excel's four-position display. Nine industry codes contain `X`, and all nine of those records have complete occupation targets. Preserve industry codes as strings and retain every recorded `X`. Industry is an existing classification whose availability before occupation prediction is not established. Prepare it separately for one-hot encoding, but exclude it from the primary feature matrix so that the primary input does not depend on an unverified upstream classification.

In [36]:
# Restore four-position numeric codes and preserve partial strings exactly.
prepared_data["SemelAnafSofi"] = industry_codes.where(~industry_codes.str.fullmatch(r"[0-9]{1,4}", na=False), industry_codes.str.zfill(4))
assert prepared_data["SemelAnafSofi"].str.fullmatch(r"[0-9X]{4}", na=False).all()
preparation_audit["Industry_partial"] = prepared_data["SemelAnafSofi"].str.contains("X", na=False)
industry_input = prepared_data[["SemelAnafSofi"]].copy()
display(prepared_data["SemelAnafSofi"].value_counts().rename("Records"))


SemelAnafSofi
5610    14
6201    12
8411    11
6419    11
5510     9
4711     8
4772     8
4770     8
8010     7
8423     7
6202     6
XXXX     6
9000     6
8121     6
7201     6
9602     6
1071     5
8501     5
8890     5
8610     4
2610     4
0123     4
4323     4
2630     4
6203     3
8550     3
3600     3
1073     3
4327     3
1074     2
6921     2
4101     2
4523     2
8110     1
8542     1
6110     1
0124     1
8312     1
851X     1
4103     1
8311     1
852X     1
3XXX     1
5310     1
Name: Records, dtype: Int64

### Analysis of SemelMishlachSofi column

This manually assigned occupation code is the prediction target. Which labels can support supervised learning, how many classes have little support, and do identical predictor records ever carry different labels?

In [37]:
# Normalize the code representation without filling any unresolved digit.
occupation_codes = survey["SemelMishlachSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
occupation_codes = occupation_codes.where(~occupation_codes.str.fullmatch(r"[0-9]{1,4}", na=False), occupation_codes.str.zfill(4))
complete_target = occupation_codes.str.fullmatch(r"[0-9]{4}", na=False)
class_counts = occupation_codes.loc[complete_target].value_counts()
display(pd.Series({"Complete labels": int(complete_target.sum()), "Incomplete labels": int((~complete_target).sum()), "Complete classes": class_counts.size, "Singleton classes": int(class_counts.eq(1).sum())}))
display(survey.loc[~complete_target, ["ID"] + role_fields + ["SemelMishlachSofi"]])
# Detect label disagreements using original predictors rather than transformed employer values.
original_survey = data.loc[data["ID"].notna()].copy()
original_predictors = original_survey.drop(columns=["ID", "SemelMishlachSofi"]).astype("string").apply(lambda column: column.str.strip().replace("", pd.NA))
predictor_groups = original_predictors.fillna("<MISSING>").apply(tuple, axis=1)
conflicting_labels = occupation_codes.groupby(predictor_groups).transform("nunique").gt(1)
display(original_survey.loc[conflicting_labels, ["ID", "ShemAvoda"] + role_fields + ["SemelMishlachSofi"]])
display(class_counts.rename("Records"))


Complete labels      191
Incomplete labels      9
Complete classes      59
Singleton classes     24
dtype: int64

,ID,EzoAvoda,TeurPeula,TeurTafkid,SemelMishlachSofi
17,18.000000,לא יכול לפרט,nan,nan,1XXX
44,45.000000,nan,nan,nan,XXXX
49,50.000000,חסוי,nan,nan,1XXX
80,81.000000,לא יכול לפרט,nan,nan,1XXX
107,108.000000,nan,nan,nan,XXXX
112,113.000000,חסוי,nan,nan,1XXX
141,142.000000,nan,nan,nan,XXXX
164,165.000000,קבלת קהל,קבלת קהל,nan,335X
192,193.000000,nan,nan,nan,XXXX


,ID,ShemAvoda,EzoAvoda,TeurPeula,TeurTafkid,SemelMishlachSofi
1,2.000000,בית חולים הדסה עין כרם,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",3311
32,33.000000,בית חולים הדסה עין כרם,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",3211
64,65.000000,בית חולים הדסה עין כרם,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",3211
95,96.000000,בית חולים הדסה עין כרם,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",3211


SemelMishlachSofi
5223    14
9112    12
2512    10
5120     7
2511     7
5414     7
5412     7
5230     6
5322     6
2631     6
5142     6
9211     5
8183     5
2342     5
5131     5
2635     5
3512     4
3122     4
2514     4
7414     4
4313     4
2655     4
2120     3
3211     3
7127     3
2131     3
8212     2
7412     2
3323     2
3513     2
9412     2
2142     2
9329     2
2262     2
4211     2
3311     1
2113     1
4227     1
3354     1
5153     1
3332     1
2431     1
7126     1
1212     1
1330     1
1345     1
1221     1
2411     1
2423     1
7522     1
8342     1
2330     1
5222     1
2412     1
3334     1
7413     1
3312     1
5419     1
4413     1
Name: Records, dtype: Int64

**Conclusion and preparation:** There are 191 complete labels across 59 classes, including 24 singleton classes, and nine incomplete labels. Records 2, 33, 65, and 96 have identical original predictors but disagree between codes 3311 and 3211. Preserve their recorded labels and flag the disagreement, because majority frequency does not establish the correct code. Separate all nine X-coded records from labeled training inputs and retain their known prefixes. Keep identical occupational descriptions in the same future validation fold, and retain IDs only for joining results to survey records. No label is guessed or corrected during preparation.

In [38]:
# Separate complete targets from records requiring a later prediction.
assert occupation_codes.str.fullmatch(r"[0-9X]{4}", na=False).all()
prepared_data["SemelMishlachSofi"] = occupation_codes
preparation_audit["Occupation_partial"] = ~complete_target
preparation_audit["Conflicting_recorded_labels"] = conflicting_labels
training_ids = survey.loc[complete_target, "ID"].astype("int64")
prediction_ids = survey.loc[~complete_target, "ID"].astype("int64")
y_train = occupation_codes.loc[complete_target].copy()
partial_targets = occupation_codes.loc[~complete_target].copy()
# Preserve the known prefix of each incomplete target without predicting its unresolved digits.
known_prefixes = partial_targets.str.split("X").str[0]
# Group identical prepared occupational descriptions without using the target label.
occupational_profiles = prepared_data[role_fields].fillna("").apply(lambda row: tuple(str(value).strip().casefold() for value in row), axis=1)
validation_groups = pd.Series(pd.factorize(occupational_profiles, sort=False)[0], index=survey.index, name="Validation_group")
display(pd.DataFrame({"ID": prediction_ids, "Recorded target": partial_targets, "Known prefix": known_prefixes}))


,ID,Recorded target,Known prefix
17,18,1XXX,1
44,45,XXXX,
49,50,1XXX,1
80,81,1XXX,1
107,108,XXXX,
112,113,1XXX,1
141,142,XXXX,
164,165,335X,335
192,193,XXXX,


## Assemble the prepared inputs

The preparation decisions retain seven separate text fields, five categorical fields, and two numerical measurements. Missing, withheld, unresolved, and resolved text states remain distinct. ID is a join key, the occupation code is the target, and neither enters the predictors. Role-start date and industry code remain available separately for the reasons established above.

The existing workplace transformation is retained. For locality, the two recorded spellings `קרית גת` and `קריית גת` are represented by `קריית גת`. Lowercasing occurs inside TF-IDF, after the preceding LaBSE comparison. Each text field uses its own vocabulary of character sequences of length two to four, unsmoothed IDF, raw term counts, and L2 normalization. Character sequences represent recurring parts of Hebrew and English words without requiring a tokenizer for either language. Their predictive value remains to be tested later. Categorical inputs use one-hot encoding. Numerical inputs use training-set median imputation and standard scaling, with a separate missing indicator for each measurement even when that measurement is complete in the training subset.

In [39]:
# Keep the existing workplace transformation and standardize only surrounding whitespace.
prepared_data["ShemAvoda"] = text_values["ShemAvoda"].fillna("")
preparation_audit["ShemAvoda_state"] = response_states["ShemAvoda"]
# Merge the two equivalent locality spellings without altering the original survey.
prepared_data["YeshuvAvoda"] = survey["YeshuvAvoda"].astype("string").str.strip().replace({"קרית גת": "קריית גת"}).fillna("Missing")
# Preserve response states as categorical predictors alongside the prepared text.
state_columns = [field + "_state" for field in text_fields]
prepared_data[state_columns] = preparation_audit[state_columns]
categorical_fields = ["YeshuvAvoda", "MenahelEtMi", "MaamadAvoda", "MakorSachar", "TeudaGvoha"] + state_columns
numeric_fields = ["shnotlimud", "Gil"]
predictor_fields = text_fields + categorical_fields + numeric_fields
X_train = prepared_data.loc[complete_target, predictor_fields].copy()
X_prediction = prepared_data.loc[~complete_target, predictor_fields].copy()
assert set(excluded_predictors).isdisjoint(predictor_fields)
assert X_train.index.equals(y_train.index)
assert X_prediction.index.equals(partial_targets.index)
display(pd.DataFrame({"Records": [len(X_train), len(X_prediction)], "Predictor columns": [X_train.shape[1], X_prediction.shape[1]]}, index=["Complete target", "Incomplete target"]))


,Records,Predictor columns
Complete target,191,21
Incomplete target,9,21


### Encode the prepared values

Fit the feature encoder on the 191 complete-label records and transform the nine incomplete-label records without learning from them. This creates numerical inputs for a later final fit. Future cross-validation must clone and refit the entire encoder within each training fold, rather than reusing these already fitted matrices. The workplace matching above used the available survey's name vocabulary, so a later evaluation of that learned matching step must also build its candidate vocabulary inside the training fold. This section performs preparation only.

In [40]:
# Import the encoders and numerical preprocessing components.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer, MissingIndicator
from scipy import sparse
# Give each text field an independent vocabulary and L2-normalized TF-IDF representation.
text_encoders = [(field, TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), lowercase=True, smooth_idf=False, sublinear_tf=False, norm="l2", dtype=np.float64), field) for field in text_fields]
# Encode categories without imposing an order and handle categories absent from training.
category_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
# Learn numeric replacement values and scaling parameters only from training records.
numeric_encoder = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
feature_encoder = ColumnTransformer(text_encoders + [("categories", category_encoder, categorical_fields), ("numeric", numeric_encoder, numeric_fields), ("numeric_missing", MissingIndicator(features="all"), numeric_fields)], remainder="drop", sparse_threshold=1.0)
# Fit on complete-label rows and transform incomplete-label rows with the same fitted encoder.
training_matrix = sparse.csr_matrix(feature_encoder.fit_transform(X_train))
prediction_matrix = sparse.csr_matrix(feature_encoder.transform(X_prediction))
feature_names = feature_encoder.get_feature_names_out()
# Prepare industry separately so its later use requires an explicit availability decision.
industry_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
industry_training_matrix = industry_encoder.fit_transform(industry_input.loc[complete_target])
industry_prediction_matrix = industry_encoder.transform(industry_input.loc[~complete_target])
# Verify matrix alignment, numerical validity, and preservation of all source records.
assert training_matrix.shape[0] == len(y_train) and prediction_matrix.shape[0] == len(partial_targets)
assert training_matrix.shape[1] == prediction_matrix.shape[1] == len(feature_names)
assert np.isfinite(training_matrix.data).all() and np.isfinite(prediction_matrix.data).all()
assert np.array_equal(prediction_matrix[:, feature_encoder.output_indices_["numeric_missing"]].toarray(), X_prediction[numeric_fields].isna().to_numpy())
assert len(X_train) + len(X_prediction) == len(survey)
assert data.loc[survey.index, "ShemAvoda"].equals(original_survey["ShemAvoda"])
display(pd.DataFrame({"Rows": [training_matrix.shape[0], prediction_matrix.shape[0]], "Encoded features": [training_matrix.shape[1], prediction_matrix.shape[1]], "Nonzero values": [training_matrix.nnz, prediction_matrix.nnz]}, index=["Training inputs", "Prediction inputs"]))
display(pd.DataFrame({"Measurement": numeric_fields, "Training median": feature_encoder.named_transformers_["numeric"].named_steps["impute"].statistics_}))


,Rows,Encoded features,Nonzero values
Training inputs,191,6506,28846
Prediction inputs,9,6506,446


,Measurement,Training median
0,shnotlimud,14.000000
1,Gil,37.000000


### Preparation checks and saved outputs

The numerical inputs are prepared, but the sample is not certified free of coding or response errors. The audit retains the salary conflicts, title disagreement, education review cases, incomplete industry codes, and conflicting recorded occupation labels. The workplace substitution for record 140 also changes an IT-consultancy description into a cosmetics-related description despite a score above the threshold. Preserve the approved transformation result and the original text, and expose that disagreement for correction rather than treating the similarity score as evidence of employer identity.

Save the prepared tables, numerical matrices, feature names, row identifiers, reference-resolution states, and encoder. Stop here. No occupation classifier is fitted, no validation metric is reported, and no occupation code is predicted.

In [41]:
# Record the observed workplace inconsistency without overwriting the preceding transformation.
preparation_audit["Workplace_transformation_review"] = survey["ID"].eq(140)
preparation_audit["No_occupational_text"] = prepared_data[role_fields].eq("").all(axis=1)
preparation_audit["Validation_group"] = validation_groups
# Save the prepared artifacts with their survey identifiers.
import joblib
output = Path("data_preparation_results")
output.mkdir(exist_ok=True)
prepared_data.to_csv(output / "prepared_survey.csv", index=False, encoding="utf-8-sig")
preparation_audit.to_csv(output / "preparation_audit.csv", index=False, encoding="utf-8-sig")
pd.concat([training_ids.rename("ID"), X_train, y_train.rename("SemelMishlachSofi")], axis=1).to_csv(output / "training_inputs.csv", index=False, encoding="utf-8-sig")
pd.concat([prediction_ids.rename("ID"), X_prediction, partial_targets.rename("Recorded_target"), known_prefixes.rename("Known_prefix")], axis=1).to_csv(output / "prediction_inputs.csv", index=False, encoding="utf-8-sig")
sparse.save_npz(output / "training_features.npz", training_matrix)
sparse.save_npz(output / "prediction_features.npz", prediction_matrix)
sparse.save_npz(output / "industry_training_features.npz", industry_training_matrix)
sparse.save_npz(output / "industry_prediction_features.npz", industry_prediction_matrix)
pd.Series(feature_names, name="Feature").to_csv(output / "feature_names.csv", index=False, encoding="utf-8-sig")
pd.Series(industry_encoder.get_feature_names_out(), name="Feature").to_csv(output / "industry_feature_names.csv", index=False, encoding="utf-8-sig")
review.to_csv(output / "workplace_transformations.csv", index=False, encoding="utf-8-sig")
joblib.dump({"features": feature_encoder, "industry": industry_encoder}, output / "feature_encoders.joblib")
# Record provenance and the boundary between prepared inputs and later modeling.
import sklearn
manifest = {"workbook_sha256": hashlib.sha256(raw).hexdigest(), "records": len(survey), "training_records": len(X_train), "prediction_records": len(X_prediction), "features": len(feature_names), "predictor_fields": predictor_fields, "excluded_primary_fields": excluded_predictors, "training_ids": training_ids.tolist(), "prediction_ids": prediction_ids.tolist(), "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__, "classifier_trained": False, "cross_validation": "Refit every learned preparation step within each training fold.", "workplace_matching": "Preserved preceding full-survey LaBSE transformation. Record 140 requires review."}
(output / "preparation_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(preparation_audit.select_dtypes(include="bool").sum().rename("Flagged records"))
print("Data preparation complete. No occupation classifier was trained.")


Data preparation complete. No occupation classifier was trained.


Title_work_disagreement              1
MenahelEtMi_missing                168
Salary_eligibility_conflict          9
Qualification_schooling_review       2
shnotlimud_missing                   2
shnotlimud_invalid                   0
Schooling_exceeds_age                0
Gil_missing                          2
Gil_invalid                          0
TarichHathala_invalid                0
Industry_partial                     9
Occupation_partial                   9
Conflicting_recorded_labels          4
Workplace_transformation_review      1
No_occupational_text                10
Name: Flagged records, dtype: Int64